# 04 — Fine-tune YOLO

The "production" detector from the project brief: fine-tune a pretrained YOLO11n on the same 19 clean
scenes, using the same scene-level 5-fold split as notebook 03, so the comparison in notebook 05 is
apples-to-apples. This stage is intentionally framework-mediated (`ultralytics`) — notebook 03 already
made every mechanic (sliding window, IoU, NMS) explicit by hand, so here the goal is a strong, fast,
realistic result to honestly compare against that from-scratch baseline, not to re-derive YOLO's
internals.

To keep the comparison fair, we evaluate YOLO's predictions with the *same* `match_detections` /
`precision_recall` functions from `src/eval/metrics.py` that scored the sliding-window model — not
ultralytics' own mAP calculation, which uses a different matching procedure.

In [ ]:
import ast
import shutil
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from ultralytics import YOLO

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.yolo_export import export_yolo_fold
from src.eval.metrics import match_detections, mean_localization_error, precision_recall

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RUNS_DIR = PROJECT_ROOT / "runs"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)
plt.rcParams["figure.dpi"] = 110

scenes = pd.read_csv(PROCESSED_DIR / "clean_manifest.csv")
scenes["boxes"] = scenes["boxes"].apply(ast.literal_eval)
scenes["scene_id"] = scenes["scene_id"].astype(str)
fold_assignment = pd.read_csv(PROCESSED_DIR / "fold_assignment.csv")
fold_assignment["scene_id"] = fold_assignment["scene_id"].astype(str)
N_FOLDS = fold_assignment["val_fold"].nunique()
IOU_THRESHOLD = 0.3

## Export fold 0 to YOLO format

`export_yolo_fold` (`src/data/yolo_export.py`) writes `train/{images,labels}` and `val/{images,labels}`
folders plus a `data.yaml`, converting our absolute-pixel `(x1,y1,x2,y2)` boxes to YOLO's normalized
`(class, cx, cy, w, h)` format.

In [ ]:
fold0_dir = PROCESSED_DIR / "yolo_fold0"
data_yaml_fold0 = export_yolo_fold(scenes, fold_assignment, val_fold=0, out_dir=fold0_dir)
print(data_yaml_fold0.read_text())

sample_label = next((fold0_dir / "train" / "labels").glob("*.txt"))
print(f"\nExample label ({sample_label.name}): {sample_label.read_text()}")

## Train fold 0 in detail

Starting from pretrained `yolo11n.pt` (nano — smallest YOLO11 variant) rather than training from random
weights: with only 15 training scenes, training from scratch would badly overfit long before learning
anything general, whereas fine-tuning from COCO-pretrained weights lets the model reuse general-purpose
visual features and only adapt to "what Waldo looks like". `imgsz=640` matches our images' native
resolution exactly (recall from notebook 01: every image was resized to 640x640 during the Roboflow
export), so no further resizing distortion is introduced. Batch size is small (8) to match the small
training set; ultralytics' built-in mosaic/flip/HSV augmentation substitutes for the hand-written
augmentation pipeline notebook 02 built for the sliding-window classifier.

In [ ]:
model_fold0 = YOLO("yolo11n.pt")
results_fold0 = model_fold0.train(
    data=str(data_yaml_fold0),
    epochs=60,
    imgsz=640,
    batch=8,
    patience=20,
    project=str(RUNS_DIR),
    name="yolo_fold0",
    exist_ok=True,
    verbose=False,
    plots=True,
)

In [ ]:
results_csv = RUNS_DIR / "yolo_fold0" / "results.csv"
results_df = pd.read_csv(results_csv)
results_df.columns = [c.strip() for c in results_df.columns]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(results_df["epoch"], results_df["train/box_loss"], label="train box loss")
axes[0].plot(results_df["epoch"], results_df["val/box_loss"], label="val box loss")
axes[0].legend()
axes[0].set_title("Box regression loss")
axes[1].plot(results_df["epoch"], results_df["metrics/mAP50(B)"])
axes[1].set_title("val mAP@0.5 (ultralytics' own metric)")
plt.tight_layout()
plt.show()

## Qualitative check

In [ ]:
def show_yolo_detections(image, yolo_result, gt_boxes, title):
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(image)
    for box in gt_boxes:
        x1, y1, x2, y2 = box
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="lime", linewidth=2))
    for box in yolo_result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="red", linewidth=2))
        ax.text(x1, y1 - 4, f"{box.conf.item():.2f}", color="red", fontsize=8)
    ax.set_title(title, fontsize=9)
    ax.axis("off")
    plt.tight_layout()
    plt.show()


fold0_val_scenes = scenes.merge(fold_assignment, on="scene_id").query("val_fold == 0")
sample_scene = fold0_val_scenes.iloc[0]
sample_image = Image.open(sample_scene["image_path"])
result = model_fold0.predict(sample_image, conf=0.4, verbose=False)[0]
show_yolo_detections(sample_image, result, sample_scene["boxes"], f"scene {sample_scene['scene_id']}")

## Full cross-validation (our own metrics, matching notebook 03)

In [ ]:
yolo_all_matches = []
yolo_per_scene = []
yolo_fold_models = {0: model_fold0}

for fold in range(N_FOLDS):
    if fold == 0:
        fold_model = model_fold0
    else:
        fold_dir = PROCESSED_DIR / f"yolo_fold{fold}"
        data_yaml = export_yolo_fold(scenes, fold_assignment, val_fold=fold, out_dir=fold_dir)
        fold_model = YOLO("yolo11n.pt")
        fold_model.train(
            data=str(data_yaml), epochs=40, imgsz=640, batch=8, patience=15,
            project=str(RUNS_DIR), name=f"yolo_fold{fold}", exist_ok=True, verbose=False, plots=False,
        )
        yolo_fold_models[fold] = fold_model

    val_scenes = scenes.merge(fold_assignment, on="scene_id").query("val_fold == @fold")
    for _, scene in val_scenes.iterrows():
        image = Image.open(scene["image_path"])
        result = fold_model.predict(image, conf=0.4, verbose=False)[0]
        pred_boxes = [tuple(b.xyxy[0].tolist()) for b in result.boxes]
        match = match_detections(pred_boxes, scene["boxes"], iou_threshold=IOU_THRESHOLD)
        yolo_all_matches.append(match)
        yolo_per_scene.append(
            {"scene_id": scene["scene_id"], "fold": fold, "tp": match.true_positives, "fp": match.false_positives, "fn": match.false_negatives}
        )
    print(f"fold {fold} done")

yolo_precision, yolo_recall = precision_recall(yolo_all_matches)
yolo_mle = mean_localization_error(yolo_all_matches)
print(f"\nYOLO across all {len(scenes)} scenes (IoU>={IOU_THRESHOLD}):")
print(f"  Precision: {yolo_precision:.2f}")
print(f"  Recall:    {yolo_recall:.2f}")
print(f"  Mean localization error: {yolo_mle:.1f}px" if yolo_mle else "  No true positives")

pd.DataFrame(yolo_per_scene)

In [ ]:
yolo_results_out = pd.DataFrame(yolo_per_scene)
yolo_results_out.to_csv(PROCESSED_DIR / "yolo_cv_results.csv", index=False)
print(f"Saved YOLO CV results to {PROCESSED_DIR / 'yolo_cv_results.csv'}")

## Inference speed: sliding window vs. single forward pass

Notebook 03 flagged this as worth measuring directly: exhaustive multi-scale sliding-window inference
runs the classifier on thousands of crops per image, while YOLO does one forward pass over the whole
image at once.

In [ ]:
n_runs = 3

start = time.perf_counter()
for _ in range(n_runs):
    model_fold0.predict(sample_image, conf=0.4, verbose=False)
yolo_time = (time.perf_counter() - start) / n_runs
print(f"YOLO:            {yolo_time*1000:.0f} ms / image")

try:
    from src.models.sliding_window import detect
    from src.models.patch_classifier import PatchClassifier
    import torch

    sweep_path = PROCESSED_DIR / "sliding_window_threshold_sweep.csv"
    sw_threshold = pd.read_csv(sweep_path).pipe(lambda d: d.loc[d["f1"].idxmax(), "threshold"]) if sweep_path.exists() else 0.6

    sw_model = PatchClassifier()
    sw_model.load_state_dict(torch.load(MODELS_DIR / "sliding_window_classifier.pt"))

    start = time.perf_counter()
    for _ in range(n_runs):
        detect(sw_model, sample_image, score_threshold=sw_threshold)
    sw_time = (time.perf_counter() - start) / n_runs
    print(f"Sliding window:  {sw_time*1000:.0f} ms / image  ({sw_time/yolo_time:.0f}x slower than YOLO)")
except FileNotFoundError:
    print("Sliding-window model not found yet — run notebook 03 first.")

## Save the model

In [ ]:
best_weights = RUNS_DIR / "yolo_fold0" / "weights" / "best.pt"
shutil.copy(best_weights, MODELS_DIR / "yolo_detector.pt")
print(f"Saved to {MODELS_DIR / 'yolo_detector.pt'}")

## Takeaways → what this motivates for notebook 05

- Same 19 scenes, same 5 folds, same evaluation function as notebook 03 — the precision/recall/mean
  localization error numbers above are directly comparable to the sliding-window baseline's.
- YOLO's built-in augmentation (mosaic, HSV jitter, flips) replaces the hand-written pipeline from
  notebook 02 — worth naming explicitly in the write-up as an example of what a framework automates.
- The timing comparison above quantifies the practical cost of the from-scratch approach: useful for
  understanding, expensive at inference time.

Notebook 05 puts both models' cross-validated results side by side, and does the real error analysis:
visualizing specific false positives/negatives from both models on the same scenes.